# Muninn Alpha Backtest Demo

This notebook pulls deterministic features from a running Muninn `query-api` and runs a tiny
research workflow against them: fetch features, compute forward returns from a VWAP series,
and look at the Information Coefficient (Spearman rank correlation between today's signal and
tomorrow's return).

**Prerequisites.** A Muninn server running on `localhost:8080` with the `query-api` exposing
`vwap.1m`, `obi`, and `vpin` features. See the main repo's [Quickstart](https://github.com/lgreene03/muninn#quickstart)
section. If your features aren't named exactly like this, adjust the `FEATURES` list below.

**What this demo is not.** It is not a trading strategy. It is a research-workflow
demonstration. The point is reproducibility — every value you see here was produced by code
whose output a replay will reproduce byte-for-byte (see [DETERMINISTIC_REPLAY.md](https://github.com/lgreene03/muninn/blob/main/docs/steering/DETERMINISTIC_REPLAY.md)).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import polars as pl
import seaborn as sns

from muninn import MuninnClient

sns.set_theme(style="whitegrid")
pl.Config.set_tbl_rows(8)

INSTRUMENT = "BTC-USDT"
FEATURES = ["vwap.1m", "obi", "vpin"]
START = "2026-05-10T14:00:00Z"
END = "2026-05-10T18:00:00Z"

## 1. Discover available features

`list_features` calls the server's discovery endpoint and returns the registered
definitions. Use it to confirm the names below exist on this Muninn instance.

In [ ]:
with MuninnClient() as muninn:
    definitions = muninn.list_features()

pd.DataFrame([d.model_dump() for d in definitions])

## 2. Pull a multi-feature panel

One call. The SDK fetches each feature, joins on `event_time` (outer by default), and returns
a Polars DataFrame. Convert to Pandas at the boundary with `.to_pandas()` when a library
downstream demands it.

In [ ]:
with MuninnClient() as muninn:
    panel = muninn.get_features(
        instrument=INSTRUMENT,
        features=FEATURES,
        start=START,
        end=END,
    )

panel.head()

## 3. Compute forward returns

Use the VWAP series as a price proxy and compute log returns one step forward. Real research
would use trade-print prices, but VWAP is a fine stand-in for a notebook demo.

In [ ]:
df = panel.to_pandas().set_index("event_time").sort_index()

df["log_price"] = np.log(df["vwap.1m"].astype(float))
df["fwd_return_1"] = df["log_price"].diff().shift(-1)

df[["vwap.1m", "fwd_return_1", "obi", "vpin"]].head()

## 4. Information Coefficient

Spearman rank correlation between each signal and the next-bar forward return. This is
intentionally a *toy* IC — short window, single instrument, no winsorization, no significance
test. The point is to demonstrate the data path, not to ship alpha.

In [ ]:
signals = [c for c in ("obi", "vpin") if c in df.columns]
ic = (
    df[signals + ["fwd_return_1"]]
    .dropna()
    .corr(method="spearman")["fwd_return_1"]
    .drop("fwd_return_1")
    .sort_values()
)
ic.to_frame("IC")

## 5. Plot signal vs. forward return

A quick visual: scatter each signal against the next-bar return, with a smoothed conditional
mean. Slope is a rough sanity check before any rigorous modeling.

In [ ]:
scatter_df = df.dropna(subset=["fwd_return_1"] + signals)

fig, axes = plt.subplots(1, len(signals), figsize=(6 * len(signals), 4), sharey=True)
if len(signals) == 1:
    axes = [axes]

for ax, sig in zip(axes, signals, strict=True):
    sns.regplot(
        data=scatter_df,
        x=sig,
        y="fwd_return_1",
        ax=ax,
        lowess=False,
        scatter_kws={"alpha": 0.4, "s": 12},
        line_kws={"color": "crimson"},
    )
    ax.set_title(f"{sig} vs forward return")
    ax.axhline(0, color="k", lw=0.5)

fig.tight_layout()

## 6. Reproduce: submit a replay

The signal numbers above came from the live feature pipeline. To confirm the same inputs
produce the same outputs, submit a replay job over the same range and poll until terminal.
When it completes, fetch the replayed feature series and assert they match the live ones —
the central architectural claim of Muninn.

In [ ]:
import time

with MuninnClient() as muninn:
    job = muninn.submit_replay_job(start=START, end=END, feature_version="v1")
    print(f"submitted job_id={job.job_id} status={job.status}")

    while not job.is_terminal:
        time.sleep(2)
        job = muninn.get_replay_job(job.job_id)
        print(f"...status={job.status} eventsReplayed={job.events_replayed}")

    print(f"final status: {job.status}, elapsed: {job.elapsed}")

## Next steps

- Swap `BTC-USDT` for any instrument the server has registered (`list_features` will show what's available).
- Try cross-asset features once a second adapter lands on the server side.
- Move from notebook to a script when your research stabilizes — `MuninnClient` is identical in either context.
- For a longer worked example, including divergence detection and the determinism property, see [DETERMINISTIC_REPLAY.md §Worked Example](https://github.com/lgreene03/muninn/blob/main/docs/steering/DETERMINISTIC_REPLAY.md#worked-example-vwap-over-six-trades) on the main repo.